# 🐍 Clase 7 · Del snapshot real al OrderBook

> Convertir una fila real del feed en Level y OrderBook, y ampliar su API con depth, imbalance y microprice sin duplicar lógica.

**Hoy construyes:** OrderBook: transformar datos externos en estado ordenado y consultable.

⏱️ 🟢 núcleo ~23 min · 🔵 si vamos bien +26 min.

### Cómo funciona este cuaderno

1. Escribe tu respuesta en la celda de código.
2. Debajo hay una **✅ comprobación plegada**: ejecútala con `Shift+Enter` para validarte (despliégala si quieres ver el `assert`).
3. ¿Atascado? Algunos ejercicios traen una **💭 Pista** intermedia; si no basta, abre **💡 Ver solución**.

Cada ejercicio lleva su etiqueta: **🟢 núcleo** (en clase) · **🔵 si vamos bien** · **🟣 bonus** (el cuaderno de auxiliares profundiza más).

### B1 · Construye Level

<sub>🟢 núcleo · ~3 min</sub>

Instancia `bid` y `ask` como dos `Level` independientes. Cambiar el tamaño de uno no puede afectar al otro.

<sub>practicas: dataclass e independencia de objetos</sub>

In [ ]:
from exchange.book import Level
bid = None
ask = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert bid is not None, '⏸ bid sigue en None: completa el ejercicio antes de validar'
assert ask is not None, '⏸ ask sigue en None: completa el ejercicio antes de validar'
assert isinstance(bid, Level) and isinstance(ask, Level)
assert bid is not ask
old = ask.size; bid.size += 1
assert ask.size == old, 'cada nivel mantiene su propio estado'
assert bid.price > 0 and ask.price > bid.price
print('ok ->', bid, ask)

<details>
<summary>💡 Ver solución</summary>

```python
bid = Level(100.0, 2.0)
ask = Level(101.0, 3.0)
```

</details>

### B2 · Ordena bids y asks

<sub>🟢 núcleo · ~5 min</sub>

Implementa `sort_sides`: bids de mayor a menor precio; asks de menor a mayor. No alteres las listas recibidas.

<sub>practicas: sorted(key=...)</sub>

In [ ]:
from exchange.book import Level
bids = [Level(99, 1), Level(101, 2), Level(100, 3)]
asks = [Level(103, 1), Level(101, 2), Level(102, 3)]
def sort_sides(bids, asks):
    pass

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert sort_sides.__code__.co_consts != (None,), '⏸ implementa sort_sides: su cuerpo sigue siendo pass'
b0, a0 = list(bids), list(asks)
sb, sa = sort_sides(bids, asks)
assert [x.price for x in sb] == [101, 100, 99]
assert [x.price for x in sa] == [101, 102, 103]
assert bids == b0 and asks == a0, 'devuelve listas ordenadas nuevas'
print('ok')

<details>
<summary>💡 Ver solución</summary>

```python
def sort_sides(bids, asks):
    return (sorted(bids, key=lambda lv: -lv.price),
            sorted(asks, key=lambda lv: lv.price))
```

</details>

### B3 · Raw snapshot → niveles

<sub>🟢 núcleo · ~7 min</sub>

Convierte el snapshot de tres niveles en dos listas de `Level`. Ignora tamaños nulos o no positivos.

<sub>practicas: nombres dinámicos y bucle</sub>

In [ ]:
from exchange.book import Level
row = {
 'bid_price_1': 100, 'bid_size_1': 2, 'ask_price_1': 101, 'ask_size_1': 1,
 'bid_price_2': 99,  'bid_size_2': 0, 'ask_price_2': 102, 'ask_size_2': 3,
 'bid_price_3': 98,  'bid_size_3': 4, 'ask_price_3': 103, 'ask_size_3': 2,
}
def levels_from_snapshot(row, depth):
    pass

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert levels_from_snapshot.__code__.co_consts != (None,), '⏸ implementa levels_from_snapshot: su cuerpo sigue siendo pass'
bids, asks = levels_from_snapshot(row, 3)
assert [(x.price, x.size) for x in bids] == [(100.0, 2.0), (98.0, 4.0)]
assert [(x.price, x.size) for x in asks] == [(101.0, 1.0), (102.0, 3.0), (103.0, 2.0)]
print('ok')

<details>
<summary>💭 Pista (antes de mirar la solución)</summary>

En cada `i`, construye los nombres `bid_price_{i}`, `bid_size_{i}`, etc. Añade Level solo cuando precio y tamaño existan y `size > 0`.

</details>

<details>
<summary>💡 Ver solución</summary>

```python
def levels_from_snapshot(row, depth):
    bids, asks = [], []
    for i in range(1, depth + 1):
        bp, bs = row.get(f'bid_price_{i}'), row.get(f'bid_size_{i}')
        ap, az = row.get(f'ask_price_{i}'), row.get(f'ask_size_{i}')
        if bp is not None and bs is not None and float(bs) > 0:
            bids.append(Level(float(bp), float(bs)))
        if ap is not None and az is not None and float(az) > 0:
            asks.append(Level(float(ap), float(az)))
    return bids, asks
```

</details>

### B4 · Implementa from_snapshot

<sub>🟢 núcleo · ~8 min</sub>

Completa `StudentOrderBook.from_snapshot`. Debe fabricar la instancia con `cls(...)`; `__init__` se ocupa de ordenar.

<sub>practicas: @classmethod como factory</sub>

In [ ]:
from exchange.book import Level
row = {'bid_price_1':99,'bid_size_1':1,'ask_price_1':102,'ask_size_1':2,
       'bid_price_2':100,'bid_size_2':3,'ask_price_2':101,'ask_size_2':4}
class StudentOrderBook:
    def __init__(self, symbol, bids, asks):
        self.symbol = symbol
        self.bids = sorted(bids, key=lambda lv: -lv.price)
        self.asks = sorted(asks, key=lambda lv: lv.price)

    @classmethod
    def from_snapshot(cls, symbol, row, depth=10):
        pass

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert StudentOrderBook.from_snapshot.__code__.co_consts != (None,), '⏸ implementa StudentOrderBook.from_snapshot: su cuerpo sigue siendo pass'
book = StudentOrderBook.from_snapshot('BTCUSDT', row, 2)
assert isinstance(book, StudentOrderBook) and book.symbol == 'BTCUSDT'
assert [x.price for x in book.bids] == [100.0, 99.0]
assert [x.price for x in book.asks] == [101.0, 102.0]
print('ok')

<details>
<summary>💡 Ver solución</summary>

```python
class StudentOrderBook:
    def __init__(self, symbol, bids, asks):
        self.symbol = symbol
        self.bids = sorted(bids, key=lambda lv: -lv.price)
        self.asks = sorted(asks, key=lambda lv: lv.price)

    @classmethod
    def from_snapshot(cls, symbol, row, depth=10):
        bids, asks = [], []
        for i in range(1, depth + 1):
            bp, bs = row.get(f'bid_price_{i}'), row.get(f'bid_size_{i}')
            ap, az = row.get(f'ask_price_{i}'), row.get(f'ask_size_{i}')
            if bp is not None and bs is not None and float(bs) > 0:
                bids.append(Level(float(bp), float(bs)))
            if ap is not None and az is not None and float(az) > 0:
                asks.append(Level(float(ap), float(az)))
        return cls(symbol, bids, asks)
```

</details>

### B5 · Implementa depth

<sub>🔵 si vamos bien · ~5 min</sub>

Completa `depth(side, levels)`: suma el tamaño de los primeros niveles del lado solicitado.

<sub>practicas: método de consulta</sub>

In [ ]:
from exchange.book import Level
from exchange.orders import Side
class StudentOrderBook:
    def __init__(self):
        self.bids = [Level(100,2), Level(99,3), Level(98,5)]
        self.asks = [Level(101,7), Level(102,11)]
    def depth(self, side, levels=10):
        pass
book = StudentOrderBook()

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert StudentOrderBook.depth.__code__.co_consts != (None,), '⏸ implementa StudentOrderBook.depth: su cuerpo sigue siendo pass'
assert book.depth(Side.BUY, 2) == 5
assert book.depth('sell', 1) == 7
assert book.depth(Side.SELL, 99) == 18
print('ok')

<details>
<summary>💡 Ver solución</summary>

```python
class StudentOrderBook:
    def __init__(self):
        self.bids = [Level(100,2), Level(99,3), Level(98,5)]
        self.asks = [Level(101,7), Level(102,11)]
    def depth(self, side, levels=10):
        side = Side(side)
        selected = self.bids if side is Side.BUY else self.asks
        return sum(lv.size for lv in selected[:levels])
book = StudentOrderBook()
```

</details>

### B6 · Implementa imbalance componiendo métodos

<sub>🔵 si vamos bien · ~6 min</sub>

Completa `imbalance`. No vuelvas a recorrer bids/asks: apóyate en `depth()` y trata el libro vacío.

<sub>practicas: reutilizar depth</sub>

In [ ]:
from exchange.book import Level
from exchange.orders import Side
class StudentOrderBook:
    def __init__(self, bids, asks): self.bids, self.asks = bids, asks
    def depth(self, side, levels=10):
        side = Side(side); xs = self.bids if side is Side.BUY else self.asks
        return sum(lv.size for lv in xs[:levels])
    def imbalance(self, levels=1):
        pass

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert StudentOrderBook.imbalance.__code__.co_consts != (None,), '⏸ implementa StudentOrderBook.imbalance: su cuerpo sigue siendo pass'
book = StudentOrderBook([Level(100,3)], [Level(101,1)])
assert abs(book.imbalance(1) - 0.5) < 1e-12
assert StudentOrderBook([], []).imbalance(2) is None
print('ok')

<details>
<summary>💡 Ver solución</summary>

```python
class StudentOrderBook:
    def __init__(self, bids, asks): self.bids, self.asks = bids, asks
    def depth(self, side, levels=10):
        side = Side(side); xs = self.bids if side is Side.BUY else self.asks
        return sum(lv.size for lv in xs[:levels])
    def imbalance(self, levels=1):
        bid = self.depth(Side.BUY, levels)
        ask = self.depth(Side.SELL, levels)
        total = bid + ask
        return None if total == 0 else (bid - ask) / total
```

</details>

### B7 · Implementa microprice

<sub>🔵 si vamos bien · ~5 min</sub>

Implementa `microprice`. Si falta cualquiera de los lados devuelve `None`; si existen, pondera cada precio por el tamaño contrario.

<sub>practicas: propiedad derivada del nivel 1</sub>

In [ ]:
from exchange.book import Level
class StudentOrderBook:
    def __init__(self, bids, asks): self.bids, self.asks = bids, asks
    @property
    def best_bid(self): return self.bids[0].price if self.bids else None
    @property
    def best_ask(self): return self.asks[0].price if self.asks else None
    @property
    def microprice(self):
        pass

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert StudentOrderBook.microprice.fget.__code__.co_consts != (None,), '⏸ implementa StudentOrderBook.microprice: su cuerpo sigue siendo pass'
book = StudentOrderBook([Level(100,3)], [Level(102,1)])
assert abs(book.microprice - 101.5) < 1e-12
assert StudentOrderBook([], [Level(102,1)]).microprice is None
print('ok')

<details>
<summary>💡 Ver solución</summary>

```python
class StudentOrderBook:
    def __init__(self, bids, asks): self.bids, self.asks = bids, asks
    @property
    def best_bid(self): return self.bids[0].price if self.bids else None
    @property
    def best_ask(self): return self.asks[0].price if self.asks else None
    @property
    def microprice(self):
        if not self.bids or not self.asks: return None
        bs, az = self.bids[0].size, self.asks[0].size
        return (self.best_bid * az + self.best_ask * bs) / (bs + az)
```

</details>

### B8 · Snapshot real contra la referencia

<sub>🔵 si vamos bien · ~10 min</sub>

Completa la clase y úsala sobre el primer snapshot real. `student` debe coincidir con `OrderBook` en orden, depth, imbalance y microprice.

<sub>practicas: integración y oráculo</sub>

In [ ]:
import csv
import os
import exchange

_data_path = os.path.join(os.path.dirname(exchange.__file__), '_data', 'btc_lob_snapshots.csv')
with open(_data_path, newline='') as _f:
    row = {k: float(v) for k, v in next(csv.DictReader(_f)).items()}
from exchange.book import Level, OrderBook
from exchange.orders import Side
class StudentOrderBook:
    # integra aquí from_snapshot, depth, imbalance y microprice
    pass
student = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert student is not None, '⏸ student sigue en None: completa el ejercicio antes de validar'
assert '__init__' in vars(StudentOrderBook), '⏸ StudentOrderBook está vacía: escribe su __init__ y sus métodos'
reference = OrderBook.from_snapshot('BTCUSDT', row, 5)
assert [x.price for x in student.bids] == [x.price for x in reference.bids]
assert [x.price for x in student.asks] == [x.price for x in reference.asks]
for side in (Side.BUY, Side.SELL): assert abs(student.depth(side, 5)-reference.depth(side, 5)) < 1e-9
assert abs(student.imbalance(5)-reference.imbalance(5)) < 1e-9
assert abs(student.microprice-reference.microprice) < 1e-9
print('ok — tu objeto coincide con el oráculo')

<details>
<summary>💡 Ver solución</summary>

```python
class StudentOrderBook:
    def __init__(self, symbol, bids, asks):
        self.symbol=symbol; self.bids=sorted(bids,key=lambda x:-x.price); self.asks=sorted(asks,key=lambda x:x.price)
    @classmethod
    def from_snapshot(cls, symbol, row, depth=10):
        bids=[]; asks=[]
        for i in range(1, depth+1):
            bp,bs=row.get(f'bid_price_{i}'),row.get(f'bid_size_{i}')
            ap,az=row.get(f'ask_price_{i}'),row.get(f'ask_size_{i}')
            if bp is not None and bs is not None and float(bs)>0: bids.append(Level(float(bp),float(bs)))
            if ap is not None and az is not None and float(az)>0: asks.append(Level(float(ap),float(az)))
        return cls(symbol,bids,asks)
    def depth(self, side, levels=10):
        side=Side(side); xs=self.bids if side is Side.BUY else self.asks
        return sum(x.size for x in xs[:levels])
    def imbalance(self, levels=1):
        b=self.depth(Side.BUY,levels); a=self.depth(Side.SELL,levels); t=b+a
        return None if t==0 else (b-a)/t
    @property
    def microprice(self):
        if not self.bids or not self.asks: return None
        b,a=self.bids[0],self.asks[0]
        return (b.price*a.size+a.price*b.size)/(b.size+a.size)
student = StudentOrderBook.from_snapshot('BTCUSDT', row, 5)
```

</details>

## Cierre

El CSV termina en la frontera del sistema: desde ahí, todo el motor habla con OrderBook.

Si llegas al ejercicio 3 ya tienes el núcleo. Los siguientes y los auxiliares consolidan.

**Siguiente clase:** seguimos construyendo el motor sobre esta pieza.

## 🚀 Llévatelo a un `.py`

Un notebook va genial para explorar, pero el código de verdad vive en archivos `.py` que se ejecutan enteros de una vez. Abre **`read_book.py`**: es lo que acabas de construir, ordenado y de una pieza.

Ejecútalo desde una terminal:

```bash
python read_book.py
```

…o aquí mismo, en la siguiente celda:

In [ ]:
!python read_book.py

> Es la misma pieza que vive en el paquete `exchange/` — aquí, condensada en un archivo que puedes leer de una sentada.